# Pipeline reproducible de sentimiento financiero

Este notebook aplica FinBERT a una muestra sintética versionada. Las etiquetas y los scores son salidas del modelo, no verdad observada ni predicciones de rentabilidad.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.sentiment import analyze_headlines, sentiment_summary

## Datos de demostración

La muestra es sintética y se incluye para que el análisis pueda repetirse sin depender de las noticias que Yahoo muestre cada día.

In [ ]:
news = pd.read_csv(ROOT / 'data' / 'sample_headlines.csv')
news[['published_at', 'ticker', 'title', 'relevance']]

## Inferencia con FinBERT

La primera ejecución descarga `ProsusAI/finbert`. La columna `model_score` es la salida softmax asociada a la etiqueta elegida; no debe interpretarse como certeza calibrada.

In [ ]:
results = analyze_headlines(news)
results[['title', 'relevance', 'model_label', 'model_score']]

In [ ]:
summary = sentiment_summary(results)
summary

In [ ]:
order = ['positive', 'neutral', 'negative']
colors = ['#2ca02c', '#7f7f7f', '#d62728']
counts = results['model_label'].value_counts().reindex(order, fill_value=0)
ax = counts.plot.bar(color=colors, figsize=(8, 5), rot=0)
ax.set_title('Distribución de etiquetas generadas por FinBERT')
ax.set_xlabel('Etiqueta del modelo')
ax.set_ylabel('Número de titulares')
plt.tight_layout();

## Interpretación y límites

- El análisis describe el tono que FinBERT asigna a cada titular.
- No mide exactitud porque la muestra no contiene etiquetas humanas de referencia.
- El sentimiento no equivale a dirección futura del precio ni constituye una recomendación de inversión.
- Las noticias de contexto deben analizarse por separado de las menciones directas a la empresa.